# Day 49: An AI That Writes Its Own Memory

Back on Day 5 we saw that "memory" is just resending the whole chat each call — that's **working memory**, and it vanishes the second the program stops.

Real agents do more: they decide what's worth keeping, **write it down**, and recall it in a future session. Today we give our AI real **long-term memory** with **LangMem** — and watch it sort what it learns into the types a memory system actually uses.

## Install

In [ ]:
%pip install langmem langchain-google-genai --quiet

## Setup

In [15]:
import os
from pathlib import Path
from typing import Literal
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from langmem import create_memory_manager
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv(dotenv_path='../.env')
os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY2"]
print("✅ Ready")

✅ Ready


## The 4 kinds of memory

An agent doesn't have one memory — it has four:

- **Working** — the live context it's reasoning over right now (the chat you resend — this is all Day 5 did, and it resets every run)
- **Episodic** — specific things that *happened* ("solved the stairs problem yesterday")
- **Semantic** — durable *facts* about you ("is weak at dynamic programming")
- **Procedural** — how it should *behave* ("keep answers short")

Day 5 only ever used working memory. Today we build the three that **last**.

## 1. The AI decides what to remember

LangMem reads a conversation and extracts **atomic memories**, each tagged by type. We don't store the raw chat — we store the distilled, classified memories.

In [16]:

class MemoryItem(BaseModel):
    content: str = Field(description="one atomic memory about the user")
    type: Literal["semantic", "episodic", "procedural"] = Field(
        description="semantic = a durable fact about who the user is; "
                    "episodic = a specific thing that happened at a time; "
                    "procedural = a preference/rule for how the assistant should respond")

manager = create_memory_manager(
    "google_genai:gemini-3.5-flash",
    schemas=[MemoryItem],
    instructions=("Extract ALL distinct memories about the user, one atomic item each. "
                  "Capture their name/background (semantic), ANY stated preference about how answers "
                  "should be given (procedural), and ANY specific event they mention doing (episodic). "
                  "Do not merge items. Do not skip preferences."))

conversation = [
    {"role": "user", "content": "Hey, I'm Yash, a backend dev. I keep struggling with dynamic programming, and please always keep your explanations short."},
    {"role": "assistant", "content": "Got it, Yash — short answers."},
    {"role": "user", "content": "Yesterday I finally solved the climbing-stairs problem."},
]

memories = manager.invoke({"messages": conversation})

for m in memories:
    print(f"[{m.content.type:<10}] {m.content.content}")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


[semantic  ] User's name is Yash.
[semantic  ] User is a backend developer.
[semantic  ] User struggles with dynamic programming.
[procedural] User wants explanations to always be kept short.
[episodic  ] User solved the climbing-stairs problem.


## 2. It writes its own memory to disk

Where should those memories live? The 2026 frontier is almost funny: **Letta** gives agents a git-backed memory *filesystem*, **Evermind** stores it as markdown, and **Arize** reported a plain markdown file beat their vector DB (exact + cheap). So we do the same — the AI writes a `memory.md` it owns, grouped by type.

In [17]:
MEMORY = Path("day49_memory.md")
sections = [("semantic", "Facts about you (semantic)"),
            ("episodic", "Things that happened (episodic)"),
            ("procedural", "How to help you (procedural)")]

lines = ["# 🧠 What the AI remembers about you\n"]
for key, title in sections:
    items = [m.content.content for m in memories if m.content.type == key]
    if items:
        lines.append(f"## {title}")
        lines += [f"- {x}" for x in items]
        lines.append("")

MEMORY.write_text("\n".join(lines))
print(MEMORY.read_text())

# 🧠 What the AI remembers about you

## Facts about you (semantic)
- User's name is Yash.
- User is a backend developer.
- User struggles with dynamic programming.

## Things that happened (episodic)
- User solved the climbing-stairs problem.

## How to help you (procedural)
- User wants explanations to always be kept short.



## 3. A new session — it reads its memory back

Imagine closing the laptop and coming back tomorrow. All that survives is `day49_memory.md`. We load **only that file** into a fresh model — no chat history — and it still knows you.

In [18]:
chat = ChatGoogleGenerativeAI(model="gemini-3.5-flash")
memory = MEMORY.read_text()

system = ("You are a study buddy. Here is your long-term memory about the user:\n\n"
          f"{memory}\n\nUse it to personalize how you help.")

reply = chat.invoke([("system", system),
                     ("human", "Quiz me on what I should practice. Keep it short.")])
print("🤖", reply.text.strip())

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


🤖 Hey Yash! Let's build on your *Climbing Stairs* win with a quick Dynamic Programming question.

**House Robber Quiz:**
You want to maximize loot from a row of houses (`nums`), but you can't rob two adjacent houses. 

What is the recurrence relation for the max loot at house `i`?

**A)** `dp[i] = max(dp[i-1], dp[i-2] + nums[i])`  
**B)** `dp[i] = dp[i-1] + dp[i-2]`  
**C)** `dp[i] = dp[i-1] + nums[i]`  

Which option is it?


## Key Takeaways

1. **Memory lives around the model, not in it.** Working memory (Day 5) resets every run; long-term memory is *extracted*, *stored*, and *recalled* — so it survives.
2. **Four types:** working (live context), episodic (events), semantic (facts), procedural (behavior). **LangMem** extracts and classifies the three lasting ones for you, instead of hoarding the raw transcript.
3. **Storage can be dead simple** — a markdown file survives restarts and is what leaders (Letta's MemFS, Evermind) actually ship. At scale, back it with a vector store so recall stays *relevant* — that's the job of the Day 50 capstone.

---

Next: Day 50 — AI Study Buddy (capstone)